# Notebook 01 — Validation on Synthetic Data

Run all three policies on a 7-day synthetic window and check the inequality
$V_\text{greedy} \leq V_\text{MPC} \leq V_\text{PF-LP}$.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

from src.eval.benchmark import run_benchmark
from src.forecasting.xgboost_forecaster import XGBoostLMPForecaster
from src.policies.mpc import solve_mpc
from src.policies.myopic_greedy import solve_myopic_greedy
from src.policies.perfect_foresight_lp import solve_perfect_foresight
from src.utils.config import DEFAULT_BATTERY
from src.utils.synthetic_data import generate_synthetic_dataset

In [ ]:
data = generate_synthetic_dataset(n_days=7, freq_minutes=5, seed=42)
data.head()

In [ ]:
fc = XGBoostLMPForecaster()
metrics = fc.fit(data['lmp'])
print(metrics)

In [ ]:
bench = run_benchmark(
    data,
    policies={
        'perfect_foresight_lp': solve_perfect_foresight,
        'myopic_greedy': solve_myopic_greedy,
        'mpc_xgboost': solve_mpc,
    },
    extra_kwargs={
        'mpc_xgboost': dict(forecaster=fc, horizon_steps=48, resolve_every=12),
    },
)
bench.table

In [ ]:
v_pf  = bench.table.set_index('policy').loc['perfect_foresight_lp', 'total_revenue']
v_g   = bench.table.set_index('policy').loc['myopic_greedy', 'total_revenue']
v_mpc = bench.table.set_index('policy').loc['mpc_xgboost', 'total_revenue']
assert v_g <= v_mpc <= v_pf + 1e-6, 'Sanity check violated'
print(f'Greedy: ${v_g:,.0f}\nMPC:    ${v_mpc:,.0f}\nPF-LP:  ${v_pf:,.0f}')

In [ ]:
# Quick visual: SOC trajectories
fig, ax = plt.subplots(figsize=(10, 3.5))
for label, res in bench.dispatch_results.items():
    ax.plot(res.schedule.index.tz_convert('America/New_York'), res.schedule['E_mwh'], label=label, linewidth=0.9)
ax.set_xlabel('Time (ET)'); ax.set_ylabel('SOC (MWh)'); ax.legend()
ax.set_title('SOC trajectory by policy (synthetic 7-day window)')
plt.show()